In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('classification').getOrCreate()

In [3]:
df=spark.read.csv('wine_data.csv',header=True,inferSchema=True)

In [4]:
df.printSchema()

root
 |-- Alcohol: double (nullable = true)
 |-- Malic_Acid: double (nullable = true)
 |-- Ash: double (nullable = true)
 |-- Ash_Alcanity: double (nullable = true)
 |-- Magnesium: integer (nullable = true)
 |-- Total_Phenols: double (nullable = true)
 |-- Flavanoids: double (nullable = true)
 |-- Nonflavanoid_Phenols: double (nullable = true)
 |-- Proanthocyanins: double (nullable = true)
 |-- Color_Intensity: double (nullable = true)
 |-- Hue: double (nullable = true)
 |-- OD280: double (nullable = true)
 |-- Proline: integer (nullable = true)
 |-- Clase: integer (nullable = true)



In [6]:
df.show(5)

+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+
|Alcohol|Malic_Acid| Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity| Hue|OD280|Proline|Clase|
+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+
|  14.23|      1.71|2.43|        15.6|      127|          2.8|      3.06|                0.28|           2.29|           5.64|1.04| 3.92|   1065|    1|
|   13.2|      1.78|2.14|        11.2|      100|         2.65|      2.76|                0.26|           1.28|           4.38|1.05|  3.4|   1050|    1|
|  13.16|      2.36|2.67|        18.6|      101|          2.8|      3.24|                 0.3|           2.81|           5.68|1.03| 3.17|   1185|    1|
|  14.37|      1.95| 2.5|        16.8|      113|         3.85|      3.49|               

In [10]:
df.summary().show()

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------------------+------------------+-----------------+-------------------+------------------+-----------------+------------------+
|summary|           Alcohol|        Malic_Acid|               Ash|     Ash_Alcanity|         Magnesium|     Total_Phenols|        Flavanoids|Nonflavanoid_Phenols|   Proanthocyanins|  Color_Intensity|                Hue|             OD280|          Proline|             Clase|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------------------+------------------+-----------------+-------------------+------------------+-----------------+------------------+
|  count|               178|               178|               178|              178|               178|               178|               178|                 178|          

In [11]:
df.count() #cuenta la cantidad de filas

178

In [13]:
len(df.columns) #Cantidad de columnas

14

In [14]:
df.groupBy('Clase').count().show() #ver el balance de las clases

+-----+-----+
|Clase|count|
+-----+-----+
|    1|   59|
|    3|   48|
|    2|   71|
+-----+-----+



In [16]:
#conteo de los nulos
from pyspark.sql.functions import col, sum, when

df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).show()

+-------+----------+---+------------+---------+-------------+----------+--------------------+---------------+---------------+---+-----+-------+-----+
|Alcohol|Malic_Acid|Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity|Hue|OD280|Proline|Clase|
+-------+----------+---+------------+---------+-------------+----------+--------------------+---------------+---------------+---+-----+-------+-----+
|      0|         0|  0|           0|        0|            0|         0|                   0|              0|              0|  0|    0|      0|    0|
+-------+----------+---+------------+---------+-------------+----------+--------------------+---------------+---------------+---+-----+-------+-----+



In [ ]:
#completar nas
df.fillna({'Alcohol':13})

In [17]:
#crear o eliminar columnas
df1=df.withColumn('acid',df['Malic_acid']+df['Ash'])
df1.show(5)

+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+-----------------+
|Alcohol|Malic_Acid| Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity| Hue|OD280|Proline|Clase|             acid|
+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+-----------------+
|  14.23|      1.71|2.43|        15.6|      127|          2.8|      3.06|                0.28|           2.29|           5.64|1.04| 3.92|   1065|    1|4.140000000000001|
|   13.2|      1.78|2.14|        11.2|      100|         2.65|      2.76|                0.26|           1.28|           4.38|1.05|  3.4|   1050|    1|             3.92|
|  13.16|      2.36|2.67|        18.6|      101|          2.8|      3.24|                 0.3|           2.81|           5.68|1.03| 3.17|   1185|    1

In [18]:
df1.drop('acid').show(5)

+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+
|Alcohol|Malic_Acid| Ash|Ash_Alcanity|Magnesium|Total_Phenols|Flavanoids|Nonflavanoid_Phenols|Proanthocyanins|Color_Intensity| Hue|OD280|Proline|Clase|
+-------+----------+----+------------+---------+-------------+----------+--------------------+---------------+---------------+----+-----+-------+-----+
|  14.23|      1.71|2.43|        15.6|      127|          2.8|      3.06|                0.28|           2.29|           5.64|1.04| 3.92|   1065|    1|
|   13.2|      1.78|2.14|        11.2|      100|         2.65|      2.76|                0.26|           1.28|           4.38|1.05|  3.4|   1050|    1|
|  13.16|      2.36|2.67|        18.6|      101|          2.8|      3.24|                 0.3|           2.81|           5.68|1.03| 3.17|   1185|    1|
|  14.37|      1.95| 2.5|        16.8|      113|         3.85|      3.49|               

In [20]:
#Preprocesamiento
from pyspark.ml.feature import VectorAssembler
X_assembler=VectorAssembler(inputCols=['Alcohol','Malic_Acid','Ash','Ash_Alcanity','Magnesium','Total_Phenols','Flavanoids','Nonflavanoid_Phenols','Proanthocyanins','Color_Intensity','Hue','OD280','Proline'],outputCol='X')

In [22]:
df1=X_assembler.transform(df).select('X','Clase')
df1.show(5)

+--------------------+-----+
|                   X|Clase|
+--------------------+-----+
|[14.23,1.71,2.43,...|    1|
|[13.2,1.78,2.14,1...|    1|
|[13.16,2.36,2.67,...|    1|
|[14.37,1.95,2.5,1...|    1|
|[13.24,2.59,2.87,...|    1|
+--------------------+-----+
only showing top 5 rows


In [23]:
#validación cruzada
train_df, test_df=df1.randomSplit([0.7,0.3])

In [25]:
test_df.show(5)

+--------------------+-----+
|                   X|Clase|
+--------------------+-----+
|[11.03,1.51,2.2,2...|    2|
|[11.61,1.35,2.7,2...|    2|
|[11.62,1.99,2.28,...|    2|
|[11.65,1.67,2.62,...|    2|
|[11.66,1.88,1.92,...|    2|
+--------------------+-----+
only showing top 5 rows


In [27]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf=RandomForestClassifier(labelCol='Clase', featuresCol='X')
modelo=rf.fit(train_df)
pred=modelo.transform(test_df)
pred.show(5)

+--------------------+-----+--------------------+--------------------+----------+
|                   X|Clase|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|[11.03,1.51,2.2,2...|    2|[0.0,0.0294117647...|[0.0,0.0014705882...|       2.0|
|[11.61,1.35,2.7,2...|    2|  [0.0,2.5,17.5,0.0]|[0.0,0.125,0.875,...|       2.0|
|[11.62,1.99,2.28,...|    2|[0.0,0.0294117647...|[0.0,0.0014705882...|       2.0|
|[11.65,1.67,2.62,...|    2|  [0.0,0.0,19.0,1.0]| [0.0,0.0,0.95,0.05]|       2.0|
|[11.66,1.88,1.92,...|    2|  [0.0,2.0,15.0,3.0]| [0.0,0.1,0.75,0.15]|       2.0|
+--------------------+-----+--------------------+--------------------+----------+
only showing top 5 rows


In [28]:
#Evaluar el modelo
evaluator=MulticlassClassificationEvaluator(labelCol='Clase', predictionCol='prediction', metricName='accuracy')
evaluator.evaluate(pred)


0.9607843137254902

In [30]:

#Grid-Search
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
pipeline_rf=Pipeline(stages=[X_assembler,rf])
parametros=ParamGridBuilder().addGrid(rf.maxDepth,[2,5,10]).addGrid(rf.numTrees,[10,100,200]).build()

select_model=CrossValidator(estimator=pipeline_rf,estimatorParamMaps=parametros,evaluator=evaluator,numFolds=5)
model_final=select_model.fit(df)
pred_grid=model_final.transform(df)
evaluator.evaluate(pred_grid)

1.0